In [ ]:
!pip install nc-time-axis

In [ ]:
import fsspec
import xarray as xr
import matplotlib.pyplot as plt

from scipy import ndimage
import numpy as np


# Create a filesystem instance pointing to Source Coop
fs = fsspec.filesystem('s3', anon=True, endpoint_url='https://data.source.coop')

# Define your target file path on Source Cooperative
# format: "owner/repository-name/path-to-file.ext"
gcm = 'CESM2-WACCM'
scenario = 'ssp370'
year = 2015

oceanTF_path = 'ismip/ismip7-gris-forcing/data/{:s}/{:s}/ocean-1000m/tf/tf_GrIS_{:s}_{:s}_ocean-1000m_v2_{:4d}.nc'.format(gcm, scenario, gcm, scenario, year)
runoff_path = 'ismip/ismip7-gris-forcing/data/{:s}/{:s}/SDBN1-1000m/sgd/sgd_GrIS_{:s}_{:s}_SDBN1-1000m_v2_{:4d}.nc'.format(gcm, scenario, gcm, scenario, year)

sectors_ds = xr.open_dataset('GrIS_Basins_Mouginot_extended_e01000m_v1.nc')


In [ ]:
# Alternatively, open file objects via s3fs if direct string paths fail with your engine
ocean_path_template = 'ismip/ismip7-gris-forcing/data/{:s}/{:s}/ocean-1000m/tf/tf_GrIS_{:s}_{:s}_ocean-1000m_v2_{:4d}.nc'
file_objs = [fs.open(f) for f in [ocean_path_template.format(gcm, scenario, gcm, scenario, year) for year in range(2015,2101)]]

# Open and combine multiple NetCDF files in parallel using Dask
oceanTF_ds = xr.open_mfdataset(
    file_objs,  # or pass file_objs
    engine="h5netcdf",  # h5netcdf is recommended for cloud/S3 netCDF reads
    combine="nested",  # or "nested" depending on your layout
    concat_dim="time",  # specify dimension to concatenate along
    parallel=True,  # requires dask installed for parallel loading
)

# Alternatively, open file objects via s3fs if direct string paths fail with your engine
runoff_path_template = 'ismip/ismip7-gris-forcing/data/{:s}/{:s}/SDBN1-1000m/sgd/sgd_GrIS_{:s}_{:s}_SDBN1-1000m_v2_{:4d}.nc'
file_objs = [fs.open(f) for f in [runoff_path_template.format(gcm, scenario, gcm, scenario, year) for year in range(2015,2101)]]

# Open and combine multiple NetCDF files in parallel using Dask
runoff_ds = xr.open_mfdataset(
    file_objs,  # or pass file_objs
    engine="h5netcdf",  # h5netcdf is recommended for cloud/S3 netCDF reads
    combine="nested",  # or "nested" depending on your layout
    concat_dim="time",  # specify dimension to concatenate along
    parallel=True,  # requires dask installed for parallel loading
)


runoff_ds

### Define function to calculate melt rate

In [ ]:
melt_rate_params = dict()
melt_rate_params['A']     = 0.0003
melt_rate_params['alpha'] = 0.39
melt_rate_params['B']     = 0.15
melt_rate_params['beta']  = 1.18

def melt_rate(q_sg, h, TF, params=melt_rate_params, k=1.64):
   q_m = k * (params['A'] * h * q_sg**params['alpha'] + params['B']) * TF**params['beta']
   return q_m

### Interpolate bed and submergedarea to the ISMIP grid

In [ ]:
# BedMachine
bed_ds = xr.open_dataset('/home/jovyan/Data/GreenlandBed/BedMachineGreenland-v5/BedMachineGreenland-v5.nc')

# Submerged area
submergedarea_ds = xr.open_dataset('basin_submerged_area_ismip.nc')

bed_interp_ds = bed_ds.interp_like(oceanTF_ds)
bed_interp_ds = bed_interp_ds.where(bed_interp_ds.bed < 0.)

submergedarea_interp_ds = submergedarea_ds.interp_like(oceanTF_ds)


### Calculate (linear) q_m on the ISMIP grid

Very important: runoff must be converted to a linear rate by dividing by the glacier's submerged area

AND

it must be converted to units of m d-1 because that's how the constants in the Rignot parameterization are setup.

In [ ]:
# Convert sgq (m3 s-1) to q (m d-1) by dividing y submerged area and converting the time units
q_sg = 86400. * runoff_ds.sgd.chunk('auto') / submergedarea_ds.submergedarea.chunk('auto') # m d-1

# Calculate the gridded melt rate
q_m_da = melt_rate(q_sg, -bed_interp_ds.bed, oceanTF_ds.tf) # m d-1

# Check a plot of gridded melt rate and submerged area (interpolated to the ISMIP grid)
fig, ax = plt.subplots(1,2,figsize=(10,5))
oceanTF_ds.tf.isel(time=6).plot(ax=ax[0], vmin=0, vmax=10)
submergedarea_interp_ds.submergedarea.plot(ax=ax[1])

### Aggregate linear melt rates (q_m) into melt flux (Q_m)
This is done by masking q_m to just the "terminus" grid cells, which are the ice grid cells that are neighboring the ocean.

In the processing below, an assumption must be made about the orientation of the terminus within a grid cell. This is needed in order to get the submerged area of the terminus. Three different assumptions are made to roughly bound the error in the calculation:
- "lower" bound = assume that the terminus is oriented either exactly vertically or horizontally; thus, the terminus length = dx
- "upper" bound = assume that the terminus is oriented along both the x and y faces of a grid cell; thus the terminus length = 2*dx
- "baseline"    = assume that the terminus is oriented diagonally within a grid cell; thus the terminus length = sqrt(2)*dx


In [ ]:
# Find grid cells with ocean
ocean_mask = bed_interp_ds.mask == 0

# Dilate the ocean mask, which buffers the mask by 1 pixel
dilated_mask = ndimage.binary_dilation(ocean_mask)

# Find grid cells with ice grounded below sea level in the buffered ocean mask, 
# thus identifying the grid cells that have ice next to ocean
terminus_mask = (dilated_mask) & (bed_interp_ds.mask == 2) & (bed_interp_ds.bed < 0)

# Within each terminu grid cells, estimate the map-view length of the terminus in three ways:
#  1.) Assume that the terminus is oriented either exactly vertically or horizontally; thus, the terminus length = dx (or dy)
dl_lower = bed_interp_ds.x[1]-bed_interp_ds.x[0]
#  2.) Assume that the terminus is oriented diagonally within a grid cell; thus the terminus length = sqrt(2)*dx
dl_baseline = np.sqrt(2) * (bed_interp_ds.x[1]-bed_interp_ds.x[0])
#  3.) Assume that the terminus is oriented along both the x and y faces of a grid cell; thus the terminus length = 2*dx
dl_upper = 2 * (bed_interp_ds.x[1]-bed_interp_ds.x[0])

# NOTE: The "upper" bound may not actually results in realistically high values because actual termini may be much more sinuous,
#       resulting in a length that is even higher than 2*dx.


In [ ]:
# Calculate melt flux (Q_m) across all terminus faces, sum for all grid cells, and plot the time series
submergedarea = -dl_baseline * bed_interp_ds.bed.where(terminus_mask)
Q_m = submergedarea * q_m_da # m3 d-1

# Convert the units to Gt/yr to get a more intuitive number
Q_m = 365 * 1000. * Q_m * 1e-12 # Gt y-1

# Rechunk to optimize the calculation
Q_m = Q_m.chunk('auto')

# Compute
Q_m_timeseries = Q_m.sum(dim=['x','y']).compute()


### Calculate melt flux timeseries

In [ ]:
%matplotlib inline
plt.close('all')

# Calculate upper/lower bounds
Q_m_timeseries_lower = Q_m_timeseries * dl_lower/dl_baseline
Q_m_timeseries_upper = Q_m_timeseries * dl_upper/dl_baseline

# Average by season and plot just the JJA values
Q_m_JJA = Q_m_timeseries.resample(time="QS-DEC").mean(dim="time")[2::4]
Q_m_JJA_lower = Q_m_timeseries_lower.resample(time="QS-DEC").mean(dim="time")[2::4]
Q_m_JJA_upper = Q_m_timeseries_upper.resample(time="QS-DEC").mean(dim="time")[2::4]

# Plot
plt.fill_between(Q_m_JJA.time.dt.year + 0.5, Q_m_JJA_lower, Q_m_JJA_upper, alpha=0.5) #, edgecolor=clr, facecolor=clr)
plt.ylabel('Q_m (Gt y-1)')


For some reason, even when I chunked and ran with Dask, the following calculation ran out of memory when I ran it for all time steps. The code below manually splits up the processing and writes netcdf files which contain the gridded JJA average melt fluxes.

<span style="color:red"><b>NOTE:</b> If you have already run the cell below once, you do not need to run it again. Skip to the next one.</span>

In [ ]:
%%timeit

step = 4*10
for idx in range(2,362,step):
    Q_m_JJA = Q_m.resample(time="QS-DEC").mean(dim="time").isel(time=slice(idx,idx+step,4))
    Q_m_JJA.to_netcdf('tmp{:03d}.nc'.format(idx))


Now that we have the JJA melt fluxes in netcdfs, the code below reads those files as an mfdataset and separates them by sector for plotting.

In [ ]:
Q_m_JJA_ = xr.open_mfdataset('tmp???.nc')

fig, ax = plt.subplots(2, 4, figsize=(18,10))



# Plot sector map
colors_7 = plt.cm.Set2.colors[:7]
sectors_ds.ID.plot(levels=np.arange(0.5,8.5), ax=ax[0,0], colors=colors_7, cbar_kwargs={'label': '', 'ticks': [1,2,3,4,5,6,7]})
ax[0,0].set_xticks([-500000, 0, 500000], labels=['-500', '0', '500'])
ax[0,0].set_xlabel('x [km]')
ax[0,0].set_yticks([-3.0e6, -2.5e6, -2.0e6, -1.5e6, -1.0e6], labels=['-3000', '-2500', '-2000', '-1500', '-1000'])
ax[0,0].set_ylabel('y [km]')

# Plot sector melt fluxes
for i, ID in enumerate(np.unique(sectors_ds.ID)):
    Q_m_JJA_sector = Q_m_JJA_.__xarray_dataarray_variable__.where(sectors_ds.ID == ID).sum(dim=['x','y'])
    
    # Calculate upper/lower bounds
    Q_m_JJA_sector_lower = Q_m_JJA_sector * dl_lower/dl_baseline
    Q_m_JJA_sector_upper = Q_m_JJA_sector * dl_upper/dl_baseline

    # Plot
    ax[(i+1)//4, (i+1)%4].fill_between(Q_m_JJA_.__xarray_dataarray_variable__.time.dt.year + 0.5, Q_m_JJA_sector_lower, Q_m_JJA_sector_upper, alpha=0.5) #, edgecolor=clr, facecolor=clr)
    ax[(i+1)//4, (i+1)%4].set_ylabel('Q_m (Gt y-1)')

# Save the figure
fig.savefig(
    'melt_flux_sectors.png', 
    dpi=300,          # Set resolution to 300 Dots Per Inch (ideal for printing)
    bbox_inches='tight', # Prevents labels or legends from getting cut off at edges
)

Finally, we integrate all melt fluxes across the entire ice sheet and plot.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 10))

# Calculate upper/lower bounds
Q_m_JJA_lower = Q_m_JJA_.__xarray_dataarray_variable__.sum(dim=['x','y']) * dl_lower/dl_baseline
Q_m_JJA_upper = Q_m_JJA_.__xarray_dataarray_variable__.sum(dim=['x','y']) * dl_upper/dl_baseline

ax.fill_between(Q_m_JJA_.__xarray_dataarray_variable__.time.dt.year + 0.5, Q_m_JJA_lower, Q_m_JJA_upper, alpha=0.5) #, edgecolor=clr, facecolor=clr)
ax.set_ylabel('Q_m (Gt y-1)')

fig.savefig(
    'melt_flux_GrIS.png', 
    dpi=300,          # Set resolution to 300 Dots Per Inch (ideal for printing)
    bbox_inches='tight', # Prevents labels or legends from getting cut off at edges
)
